# Projeto de Python para Finanças

In [24]:
# instalação
%pip install yfinance pandas numpy plotly nbformat

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Parte 1: Obter cotações e construção de carteira

In [25]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta

acoes = ['ITUB4.SA', 'PETR4.SA', 'VALE3.SA', 'IVVB11.SA']
indices = ['^BVSP', '^GSPC', 'BRL=X', 'GC=F']

data_inicio = datetime.now() - timedelta(days=365)
data_inicio = data_inicio.strftime('%Y-%m-%d')
data_fim = datetime.now().strftime('%Y-%m-%d')
print(data_inicio, data_fim)

def pegar_cotacoes(lista_tickers, data_inicio, data_fim):
    df = yf.download(lista_tickers, data_inicio, data_fim, auto_adjust=False) # auto_adjust -> ajuste do preço de fechamento, decontando o valor do dividendo
    df = df['Adj Close']
    df = df.ffill() # preenche as linhas vazias com o valor anterior
    df = df.dropna() # remove vazios
    return df

lista_tickers = acoes + indices
df_cotacoes = pegar_cotacoes(lista_tickers, data_inicio, data_fim)
display(df_cotacoes)

[*********************100%***********************]  8 of 8 completed

2025-03-20 2026-03-20


Ticker,BRL=X,GC=F,ITUB4.SA,IVVB11.SA,PETR4.SA,VALE3.SA,^BVSP,^GSPC
Date,,,,,,,,
2025-03-20,5.6481,3040.000000,28.513966,359.200012,32.541595,52.548553,131955.0,5662.890137
2025-03-21,5.6739,3018.199951,28.576185,362.649994,33.044449,52.741337,132345.0,5667.560059
2025-03-24,5.7296,3013.100098,28.602846,371.000000,32.999554,52.465931,131321.0,5767.569824
2025-03-25,5.7608,3023.699951,28.833952,368.649994,33.259956,52.640354,132068.0,5776.649902
2025-03-26,5.6984,3020.899902,28.522852,366.450012,33.574238,52.961666,132520.0,5712.200195
...,...,...,...,...,...,...,...,...
2026-03-16,5.3291,4994.000000,42.647678,394.950012,45.860001,78.839996,179875.0,6699.379883
2026-03-17,5.2334,5001.000000,42.360054,393.149994,46.380001,78.959999,180410.0,6716.089844
2026-03-18,5.1929,4889.899902,41.933578,391.100006,47.000000,77.129997,179640.0,6624.700195


In [26]:
# carteira em termos de valores financeiros
dic_carteira = {
    'ITUB4.SA': 5000,
    'VALE3.SA': 3000,
    'PETR4.SA': 4000,
    'IVVB11.SA': 6000
}

df_carteira = pd.DataFrame()
total_investido = sum(dic_carteira.values())

print(total_investido)

for ativo in dic_carteira:
    preco_incial = df_cotacoes[ativo].iloc[0]
    qtde_acoes = dic_carteira[ativo] / preco_incial
    df_carteira[ativo] = df_cotacoes[ativo] * qtde_acoes

df_carteira['Total'] = df_carteira.sum(axis=1)

display(df_carteira)

18000


,ITUB4.SA,VALE3.SA,PETR4.SA,IVVB11.SA,Total
Date,,,,,
2025-03-20,5000.000000,3000.000000,4000.000000,6000.000000,18000.000000
2025-03-21,5010.910376,3011.006013,4061.810540,6057.627755,18141.354684
2025-03-24,5015.585440,2995.283075,4056.292043,6197.104466,18264.265024
2025-03-25,5056.110460,3005.240907,4088.300637,6157.850468,18307.502472
2025-03-26,5001.558243,3023.584625,4126.931990,6121.102446,18273.177303
...,...,...,...,...,...
2026-03-16,7478.384270,4500.980016,5637.093076,6597.160336,24213.617698
2026-03-17,7427.948571,4507.830980,5701.011326,6567.093216,24203.884093
2026-03-18,7353.164949,4403.356068,5777.221349,6532.850659,24066.593025


### Parte 2: Rentabilidade e Comparação com Benchmarks

In [27]:
df_cotacoes['SP500 (R$)'] = df_cotacoes['^GSPC'] * df_cotacoes['BRL=X']
df_cotacoes['OURO (R$)'] = df_cotacoes['GC=F'] * df_cotacoes['BRL=X']
df_cotacoes['Dólar'] = df_cotacoes['BRL=X']

df_cotacoes = df_cotacoes.drop(columns=['BRL=X', 'GC=F', '^GSPC'])
display(df_cotacoes)

Ticker,ITUB4.SA,IVVB11.SA,PETR4.SA,VALE3.SA,^BVSP,SP500 (R$),OURO (R$),Dólar
Date,,,,,,,,
2025-03-20,28.513966,359.200012,32.541595,52.548553,131955.0,31984.569211,17170.223694,5.6481
2025-03-21,28.576185,362.649994,33.044449,52.741337,132345.0,32157.169739,17124.965088,5.6739
2025-03-24,28.602846,371.000000,32.999554,52.465931,131321.0,33045.867792,17263.858177,5.7296
2025-03-25,28.833952,368.649994,33.259956,52.640354,132068.0,33278.124092,17418.930330,5.7608
2025-03-26,28.522852,366.450012,33.574238,52.961666,132520.0,32550.401711,17214.296066,5.6984
...,...,...,...,...,...,...,...,...
2026-03-16,42.647678,394.950012,45.860001,78.839996,179875.0,35701.666218,26613.526059,5.3291
2026-03-17,42.360054,393.149994,46.380001,78.959999,180410.0,35147.983702,26172.232740,5.2334
2026-03-18,41.933578,391.100006,47.000000,77.129997,179640.0,34401.406842,25392.762087,5.1929


In [28]:
def calcular_rentabilidade(df):
    retorno = df.iloc[-1] / df.iloc[0] - 1
    return retorno

display(calcular_rentabilidade(df_carteira))
display(calcular_rentabilidade(df_cotacoes))

ITUB4.SA     0.481068
VALE3.SA     0.458270
PETR4.SA     0.437545
IVVB11.SA    0.079955
Total        0.333892
dtype: float64

Ticker
ITUB4.SA      0.481068
IVVB11.SA     0.079955
PETR4.SA      0.437545
VALE3.SA      0.458270
^BVSP         0.366155
SP500 (R$)    0.078183
OURO (R$)     0.398654
Dólar        -0.075813
dtype: float64

In [29]:
df_comparacao = df_cotacoes.drop(columns=acoes)
df_comparacao['Carteira'] = df_carteira['Total']

display(df_comparacao)
print(calcular_rentabilidade(df_comparacao))

Ticker,^BVSP,SP500 (R$),OURO (R$),Dólar,Carteira
Date,,,,,
2025-03-20,131955.0,31984.569211,17170.223694,5.6481,18000.000000
2025-03-21,132345.0,32157.169739,17124.965088,5.6739,18141.354684
2025-03-24,131321.0,33045.867792,17263.858177,5.7296,18264.265024
2025-03-25,132068.0,33278.124092,17418.930330,5.7608,18307.502472
2025-03-26,132520.0,32550.401711,17214.296066,5.6984,18273.177303
...,...,...,...,...,...
2026-03-16,179875.0,35701.666218,26613.526059,5.3291,24213.617698
2026-03-17,180410.0,35147.983702,26172.232740,5.2334,24203.884093
2026-03-18,179640.0,34401.406842,25392.762087,5.1929,24066.593025


Ticker
^BVSP         0.366155
SP500 (R$)    0.078183
OURO (R$)     0.398654
Dólar        -0.075813
Carteira      0.333892
dtype: float64


In [30]:
df_comparacao = df_comparacao / df_comparacao.iloc[0] - 1
display(df_comparacao)

import plotly.express as px

grafico = px.line(df_comparacao, x=df_comparacao.index, y=df_comparacao.columns) 
grafico.update_layout(template='plotly_dark') #modo dark
grafico.show()

Ticker,^BVSP,SP500 (R$),OURO (R$),Dólar,Carteira
Date,,,,,
2025-03-20,0.000000,0.000000,0.000000,0.000000,0.000000
2025-03-21,0.002956,0.005396,-0.002636,0.004568,0.007853
2025-03-24,-0.004805,0.033182,0.005453,0.014430,0.014681
2025-03-25,0.000856,0.040443,0.014485,0.019954,0.017083
2025-03-26,0.004282,0.017691,0.002567,0.008906,0.015177
...,...,...,...,...,...
2026-03-16,0.363154,0.116215,0.549981,-0.056479,0.345201
2026-03-17,0.367209,0.098904,0.524280,-0.073423,0.344660
2026-03-18,0.361373,0.075563,0.478884,-0.080593,0.337033


### Parte 3: Análise de Risco

In [38]:
# Correlação - indica se elas tendem a variar juntas
import plotly.express as px
import numpy as np

df_cotacoes['Carteira'] = df_carteira['Total']
tabela_rentabilidade_diaria = df_cotacoes / df_cotacoes.shift(1) # Shift(1) faz referencia a linha anterior
tabela_rentabilidade_diaria = np.log(tabela_rentabilidade_diaria).dropna()
tabela_correlacao = tabela_rentabilidade_diaria.corr()

grafico_correlacao = px.imshow(df_cotacoes.corr(), text_auto=True, color_continuous_scale='greens') 
grafico_correlacao.update_layout(template='plotly_dark') #modo dark
grafico_correlacao.show()

# Variancia dos retornos diarios do ativo (aplicando a função logaritima)
tabela_volatilidade = tabela_rentabilidade_diaria.std() * np.sqrt(252)# std calculo o Desvio padrão - 252 -> media de dias uteis no ano 
display(tabela_volatilidade)

Ticker
ITUB4.SA      0.214666
IVVB11.SA     0.157986
PETR4.SA      0.242020
VALE3.SA      0.242598
^BVSP         0.157497
SP500 (R$)    0.228319
OURO (R$)     0.293371
Dólar         0.122726
Carteira      0.126083
dtype: float64

### Parte 4: Análise Técnica e Indicadores